# RNN-3 · 텍스트 · 임베딩

**한 줄 요약**

- 문장을 토큰 · 단어사전 번호 · 패딩 텐서
- 임베딩 벡터 줄로 바꿔 LSTM으로 분류하고, 처음 보는 단어 · 요약 방법
- Attention까지 작은 문장 데이터로 직접 확인함 (코드는 모두 보강)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 문장을 단어 · 글자 단위로 토큰화하고, train 문장만으로 `<pad>` 0 · `<unk>` 1이 든 단어사전을 만든다
2. 길이가 다른 문장을 `pad_sequence`로 채우고 `lengths`와 `pack_padded_sequence`로 진짜 끝을 꺼낸다
3. `nn.Embedding`이 one-hot × 가중치 표와 같음을 코드로 보이고, 연습문제 8.1 메모리 차이(166.7배)를 계산한다
4. 'A 하지만 B' 문장에서 순서 없는 평균이 찍기 수준인 이유와, 처음 보는 활용형에 글자 단위가 강한 이유를 숫자로 말한다
5. `nn.MultiheadAttention`의 출력 · 가중치 모양과 pad mask를 설명하고, RNN 계열과 차이를 말한다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">리뷰 · 댓글 · 문의 글을 긍정 · 부정 · 유형으로 분류할 때</td>
<td style="text-align: left;">토큰 → 사전 → 패딩 → Embedding → LSTM (1 ~ 3 · 5장)</td>
</tr>
<tr>
<td style="text-align: left;">학습 때 없던 신조어 · 오타 · 활용형이 들어올 때</td>
<td style="text-align: left;"><code>&lt;unk&gt;</code> · 글자 단위 (3-2)</td>
</tr>
<tr>
<td style="text-align: left;">길이가 제각각인 문장을 배치로 묶을 때</td>
<td style="text-align: left;"><code>pad_sequence</code> · <code>pack_padded_sequence</code> (1-3)</td>
</tr>
<tr>
<td style="text-align: left;">모델을 저장했다가 새 문장에 쓸 때</td>
<td style="text-align: left;">같은 사전 · 같은 토큰화 (5장)</td>
</tr>
<tr>
<td style="text-align: left;">BERT · GPT 같은 Transformer를 처음 볼 때</td>
<td style="text-align: left;">Attention 한 번 실행 (4장)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/rnn3_map.png" alt="RNN-3 학습 지도" width="733">

**예상 학습 시간** · 개념 1시간 30분 + 실습 1시간 (GPU 전체 실행 1분 이내 · CPU도 수 분)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">라이브러리 · seed · 장치 · 영화 후기 문장 데이터</td>
</tr>
<tr>
<td style="text-align: center;">1</td>
<td style="text-align: left;">문장을 숫자로 (보강)</td>
<td style="text-align: left;">문장을 어떤 단위로 자르고 번호 · 패딩 텐서로 바꾸나?</td>
</tr>
<tr>
<td style="text-align: center;">2</td>
<td style="text-align: left;">임베딩</td>
<td style="text-align: left;">번호를 one-hot 대신 Embedding으로 바꾸면 무엇이 좋아지나?</td>
</tr>
<tr>
<td style="text-align: center;">3</td>
<td style="text-align: left;">문장 분류 (보강)</td>
<td style="text-align: left;">LSTM 출력을 어떻게 요약하고, 처음 보는 단어는 어떻게 되나?</td>
</tr>
<tr>
<td style="text-align: center;">4</td>
<td style="text-align: left;">Attention 맛보기 (보강)</td>
<td style="text-align: left;">Attention은 RNN과 무엇이 다른가?</td>
</tr>
<tr>
<td style="text-align: center;">5</td>
<td style="text-align: left;">종합 실습 (보강)</td>
<td style="text-align: left;">새 문장을 예측하는 파이프라인을 처음부터 끝까지 만들려면?</td>
</tr>
<tr>
<td style="text-align: center;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

라이브러리 · seed · device와 **보강 데이터**(영화 후기 문장)를 한 번에 준비. **셀은 위에서부터 차례로 실행**.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">이름</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">처음 쓰는 곳</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>set_seed()</code> · <code>DEV</code> · <code>kpad()</code></td>
<td style="text-align: left;">seed 고정 · cuDNN 결정적 모드 · GPU 없으면 CPU · 한글 2칸 폭 줄 맞추기</td>
<td style="text-align: left;">전체</td>
</tr>
<tr>
<td style="text-align: left;"><code>make_sentences()</code></td>
<td style="text-align: left;">단문(<code>음악이 정말 좋다</code>) 반 · 대조문(<code>A 하지만 B</code>, 뒤 절이 정답) 반, 1 = 긍정 · 0 = 부정</td>
<td style="text-align: left;">0장</td>
</tr>
<tr>
<td style="text-align: left;"><code>train</code></td>
<td style="text-align: left;">학습 600문장</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;"><code>test</code></td>
<td style="text-align: left;">평가 문장 중 <strong>학습과 똑같은 문장을 뺀</strong> 처음 보는 문장</td>
<td style="text-align: left;">3-1</td>
</tr>
<tr>
<td style="text-align: left;"><code>unseen</code></td>
<td style="text-align: left;">평가 단어를 처음 보는 활용형(<code>재미있었다</code> · <code>지루했다</code> …)으로 바꾼 300문장</td>
<td style="text-align: left;">3-2</td>
</tr>
</tbody>
</table>

In [1]:
# [보강] 라이브러리 불러오기 (수업 자료에 텍스트 코드가 없어 이 노트북 코드는 모두 보강)
import random  # 파이썬 난수 (데이터 만들기)

import torch                                   # 텐서 · 자동 미분
import torch.nn as nn                          # 층: RNN · LSTM · GRU · Linear
import torch.nn.functional as F                # 함수: one_hot · cross_entropy
from torch.nn.utils.rnn import (pack_padded_sequence, pad_packed_sequence,  # 패딩 · pack 도구 (1-3)
                                pad_sequence)  # 1-3


# 재현성: seed 고정 + cuDNN 결정적 모드
SEED = 20260912                            # 수업 코드와 같은 seed
torch.backends.cudnn.deterministic = True  # GPU에서도 매번 같은 계산 결과
torch.backends.cudnn.benchmark = False     # 실행마다 계산 방법이 바뀌지 않게


def set_seed(seed=SEED):     # seed를 한 번에 고정하는 함수
    random.seed(seed)        # 파이썬 random 고정
    torch.manual_seed(seed)  # CPU · GPU 난수 모두


# GPU가 없으면 CPU로 실행
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU 없으면 CPU
print(f"torch {torch.__version__} · device {DEV.type}")             # 버전 · 장치 기록


def kpad(text, width):  # 한글은 2칸 폭으로 세어 출력 줄 맞추기
    return text + " " * (width - sum(2 if ord(ch) > 0x1100 else 1  # 모자란 폭만큼 공백
                                     for ch in text))

torch 2.11.0+cu128 · device cuda


In [2]:
# [보강] 작은 영화 후기 문장 데이터 (내려받기 없음)
SUBJ = ["영화가", "배우 연기가", "음악이", "줄거리가", "결말이", "화면이"]   # 주어 6개
ADV = ["정말", "너무", "꽤", "아주", "조금"]                                 # 꾸밈말 5개
POS = ["좋다", "재미있다", "훌륭하다", "최고다", "감동적이다"]               # 긍정 평가
NEG = ["별로다", "지루하다", "최악이다", "실망스럽다", "아쉽다"]             # 부정 평가
POS_PAST = ["좋았다", "재미있었다", "훌륭했다", "최고였다", "감동적이었다"]  # 3-2
# 처음 보는 부정 활용형
NEG_PAST = ["별로였다", "지루했다", "최악이었다", "실망스러웠다", "아쉬웠다"]


def make_sentences(n, seed, pos=POS, neg=NEG):  # 단문 반 · 'A 하지만 B' 대조문 반
    rng = random.Random(seed)                   # 이 데이터 전용 난수 (전역 seed와 분리)

    def clause(good):                                          # 주어 + 꾸밈말 + 평가
        word = rng.choice(pos if good else neg)                # 평가 단어
        return f"{rng.choice(SUBJ)} {rng.choice(ADV)} {word}"  # 주어 + 꾸밈말 + 평가

    data = []                                    # (문장, 정답, 종류) 목록
    for _ in range(n):                           # 줄 n개 만들기
        y = rng.randint(0, 1)                    # 1 = 긍정 · 0 = 부정
        if rng.random() < 0.5:                   # 반은 대조문
            data.append((f"{clause(1 - y)} 하지만 {clause(y)}", y, "대조"))  # 뒤 절이 정답
        else:
            data.append((clause(y), y, "단문"))  # 절 하나
    return data                                  # 문장 목록


# 학습 600 · 평가(학습에 있던 문장은 뺌) · 처음 보는 활용형 평가
train = make_sentences(600, seed=1)                               # 학습 600문장
seen = {t for t, _, _ in train}                                   # 학습에 나온 문장
test_all = make_sentences(300, seed=2)                            # 평가 후보 300
test = [d for d in test_all if d[0] not in seen]                  # 처음 보는 문장만
unseen = make_sentences(300, seed=3, pos=POS_PAST, neg=NEG_PAST)  # 평가 단어가 처음 보는 활용형
print(f"train {len(train)} · test {len(test)}"                    # 개수
      f" (학습과 같은 문장 {len(test_all) - len(test)}개 제외) · unseen {len(unseen)}")
for text, y, kind in train[:3]:                                   # 예시 3개
    print(f"  {kind} {y} | {text}")                               # 종류 · 정답 · 문장

train 600 · test 200 (학습과 같은 문장 100개 제외) · unseen 300
  단문 0 | 음악이 정말 별로다
  단문 1 | 화면이 아주 최고다
  대조 0 | 줄거리가 아주 좋다 하지만 영화가 아주 아쉽다


---
## 1. 문장을 숫자로 (보강)

> **이 장의 흐름**
>
> - RNN-1 · 2는 처음부터 숫자(센서 · 사인파)였다
> - 문장은 글자라서 모델에 넣기 전에 조각내고 번호를 붙여 직사각형 텐서로 만드는 단계가 필요

**파이프라인 위치** · ① 토큰화 → ② 단어사전 → ③ 번호 · 패딩

### 1-1. `A` 토큰화 단어 단위 vs 글자 단위 · `split()` · `list()` <sub>보강 · Python문법 5-1 · 강의 '개가 사람을 물었다'</sub>

> 🎯 **문장을 모델에 넣으려면 먼저 어떤 단위로 자르며, 단어 단위와 글자 단위는 무엇이 다른가?**

💡 **쉽게 말하면**

- 토큰화는 **문장을 레고 블록으로 쪼개기**.
- 단어 블록은 크고 종류가 많고, 글자 블록은 작고 종류가 적어 처음 보는 단어도 아는 블록으로 다시 조립할 수 있음.

**실행 흐름**

1. 한 문장을 단어 · 글자로 나누기
2. 같은 토큰 · 다른 순서 두 문장 비교
3. train 전체에서 종류 수 · 평균 길이

**먼저 예측** · '결말이 너무 지루하다 하지만 음악이 정말 좋다'와 앞뒤를 바꾼 문장은 토큰 모음이 같을까? — 답은 코드 실행 결과 아래에

In [3]:
# [보강] 토큰화 2가지: 단어 단위(띄어쓰기) · 글자 단위
def tok_word(text):      # 단어 단위
    return text.split()  # Python문법 5-1: 공백 기준 나누기


def tok_char(text):                     # 글자 단위
    return list(text.replace(" ", ""))  # 공백을 빼고 한 글자씩


text = train[2][0]                                                              # 대조문 하나
print("단어:", tok_word(text), len(tok_word(text)), "개")                       # 단어 토큰
print("글자:", "".join(tok_char(text))[:12], "...", len(tok_char(text)), "개")  # 글자 토큰


# 같은 토큰 · 다른 순서 → 정답이 반대 (강의 '개가 사람을 물었다')
a = "음악이 정말 좋다 하지만 결말이 너무 지루하다"                        # 뒤 절 부정 → 0
b = "결말이 너무 지루하다 하지만 음악이 정말 좋다"                        # 뒤 절 긍정 → 1
print("정렬하면 같은 토큰:", sorted(tok_word(a)) == sorted(tok_word(b)))  # 재료 비교


# 학습 데이터 전체에서 종류 수 · 평균 길이 비교
for name, tok in [("단어", tok_word), ("글자", tok_char)]:               # 두 방식
    kinds = {t for s, _, _ in train for t in tok(s)}                     # 종류 모음
    avg = sum(len(tok(s)) for s, _, _ in train) / len(train)             # 평균 길이
    print(f"{name} 단위 · 종류 {len(kinds)}개 · 문장 평균 {avg:.1f}칸")  # 비교

단어: ['줄거리가', '아주', '좋다', '하지만', '영화가', '아주', '아쉽다'] 7 개
글자: 줄거리가아주좋다하지만영 ... 19 개
정렬하면 같은 토큰: True
단어 단위 · 종류 23개 · 문장 평균 5.2칸
글자 단위 · 종류 47개 · 문장 평균 14.9칸


**결과** · 예측 정답은 같음 — 정렬하면 `True`. 그런데 정답은 반대(뒤 절이 정답)라 순서를 봐야 함

- `단어: ['줄거리가', '아주', '좋다', '하지만', '영화가', '아주', '아쉽다'] 7 개` — 띄어쓰기로 나눈 7칸
- `글자: 줄거리가아주좋다하지만영 ... 19 개` — 공백을 뺀 19글자 = 19칸 (단어 단위의 약 2.7배 길이)
- `정렬하면 같은 토큰: True` — 같은 재료 · 다른 순서인데 정답은 반대 (강의의 '개가 사람을 물었다')
- `단어 단위 · 종류 23개 · 문장 평균 5.2칸` — 줄이 짧음
- `글자 단위 · 종류 47개 · 문장 평균 14.9칸` — 이 작은 데이터는 단어가 23개뿐이라 글자 종류가 더 많지만, 실제 말뭉치는 단어 종류가 훨씬 많음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">단어 단위 <code>split()</code></td>
<td style="text-align: left;">띄어쓰기로 나눔 → 줄이 짧고 뜻 단위, 종류가 많음</td>
<td style="text-align: left;">어휘가 정해진 작은 문제 · 빠른 기준 모델</td>
</tr>
<tr>
<td style="text-align: left;">글자 단위 <code>list()</code></td>
<td style="text-align: left;">한 글자씩 → 줄이 길고 종류가 적음, 처음 보는 단어도 글자로 분해</td>
<td style="text-align: left;">활용 · 오타 · 신조어가 많은 한국어 (3-2)</td>
</tr>
<tr>
<td style="text-align: left;">부분 단어 (BPE · WordPiece)</td>
<td style="text-align: left;">자주 나오는 글자 묶음 단위, 둘의 중간</td>
<td style="text-align: left;">사전학습 언어 모델(BERT · GPT)의 기본</td>
</tr>
<tr>
<td style="text-align: left;">형태소 분석기</td>
<td style="text-align: left;"><code>좋았다</code> → <code>좋</code> + <code>았</code> + <code>다</code>처럼 뜻 단위로</td>
<td style="text-align: left;">한국어 문법 단위가 중요할 때 (설치 필요)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `split(" ")`는 공백이 두 칸이면 빈 문자열이 생김
> - 인자 없는 `split()` (Python문법 5-1)

> ✅ **한 줄 정리** · 토큰화는 문장을 모델이 셀 수 있는 조각으로 나누는 첫 단계이며, 단어 단위는 짧고 뜻이 분명하고 글자 단위는 길지만 처음 보는 단어에 강하다.

**딥러닝 코드에서는**

- 실제 프로젝트는 `tokenizer = AutoTokenizer.from_pretrained(...)`(부분 단어)를 쓰지만, 하는 일은 같음
- 문장 → 토큰 → 번호

### 1-2. `A` 단어사전과 pad · unk · `sorted(set())` · `enumerate` <sub>보강 · Python문법 2-5 · 3-4</sub>

> 🎯 **토큰을 번호로 바꾸는 단어사전은 어떻게 만들고, 사전에 없는 단어는 어떻게 처리하는가?**

💡 **쉽게 말하면**

- 단어사전은 **도서관 청구기호표**.
- 책(단어)마다 번호를 붙이고, 0번은 빈 칸(`<pad>`), 1번은 '목록에 없는 책'(`<unk>`)으로 미리 비워 둠.

<img src="images/rnn3_token_flow.png" alt="토큰 → 임베딩 흐름" width="732">

**실행 흐름**

1. train 문장으로 `sorted(set())`
2. `enumerate`로 번호 (2번부터) · 문장 · 번호
3. 단어 되돌리기
4. 사전에 없는 '좋았다'

**먼저 예측** · train에 '좋다'만 있을 때 '좋았다'는 몇 번이 될까? — 답은 코드 실행 결과 아래에

In [4]:
# [보강] train 문장만으로 단어사전: 0 = <pad>, 1 = <unk>, 나머지는 정렬 순서
def build_vocab(texts, tok):                               # train 문장 → 사전
    words = sorted({w for s in texts for w in tok(s)})     # Python문법 2-5
    vocab = {"<pad>": 0, "<unk>": 1}                       # 특수 토큰 먼저
    vocab.update({w: i + 2 for i, w in enumerate(words)})  # Python문법 3-4
    return vocab                                           # 단어 → 번호


def encode(text, vocab, tok):                                 # 문장 → 번호 목록
    return [vocab.get(w, vocab["<unk>"]) for w in tok(text)]  # 없으면 1


vocab = build_vocab([s for s, _, _ in train], tok_word)                # train으로만
itos = {i: w for w, i in vocab.items()}                                # 번호 → 단어 (되돌리기용)
print("사전 크기:", len(vocab), "| 앞 5개:", list(vocab.items())[:5])  # 사전 확인


# 문장 → 번호 → 다시 단어
ids = encode("음악이 정말 좋다", vocab, tok_word)              # 아는 단어만
print("음악이 정말 좋다 →", ids, "→", [itos[i] for i in ids])  # 번호 → 단어


# 처음 보는 단어는 <unk>(1)로
ids = encode("음악이 정말 좋았다", vocab, tok_word)              # 처음 보는 활용형
print("음악이 정말 좋았다 →", ids, "→", [itos[i] for i in ids])  # <unk>로 바뀜

사전 크기: 25 | 앞 5개: [('<pad>', 0), ('<unk>', 1), ('감동적이다', 2), ('결말이', 3), ('꽤', 4)]
음악이 정말 좋다 → [13, 15, 17] → ['음악이', '정말', '좋다']
음악이 정말 좋았다 → [13, 15, 1] → ['음악이', '정말', '<unk>']


**결과** · 예측 정답은 `1` — `<unk>`. `vocab.get(단어, 1)`

- `사전 크기: 25 | 앞 5개: [('<pad>', 0), ('<unk>', 1), ('감동적이다', 2), ('결말이', 3), ('꽤', 4)]` — 특수 토큰 2개 + 단어 23개, 단어는 정렬 순서로 2번부터
- `음악이 정말 좋다 → [13, 15, 17] → ['음악이', '정말', '좋다']` — 번호 ↔ 단어를 양쪽으로 바꿀 수 있음
- `음악이 정말 좋았다 → [13, 15, 1] → ['음악이', '정말', '<unk>']` — 사전에 없는 '좋았다'는 1(`<unk>`) — 평가 단어가 통째로 사라짐

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>&lt;pad&gt;</code> (0번)</td>
<td style="text-align: left;">길이를 맞추려 채운 빈 칸</td>
<td style="text-align: left;">패딩 · <code>padding_idx=0</code> · mask</td>
</tr>
<tr>
<td style="text-align: left;"><code>&lt;unk&gt;</code> (1번)</td>
<td style="text-align: left;">사전에 없는 단어를 모두 하나로</td>
<td style="text-align: left;">처음 보는 단어가 들어와도 에러 없이 처리</td>
</tr>
<tr>
<td style="text-align: left;">train으로만 사전 만들기</td>
<td style="text-align: left;">test 단어는 사전에 넣지 않음</td>
<td style="text-align: left;"><strong>항상</strong> (test 정보 누수 방지)</td>
</tr>
<tr>
<td style="text-align: left;"><code>sorted(set())</code> + <code>enumerate</code></td>
<td style="text-align: left;">정렬로 번호가 실행마다 같음</td>
<td style="text-align: left;">저장한 모델과 사전 번호가 어긋나지 않게</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - test까지 합쳐 사전을 만들면 처음 보는 단어가 없는 것처럼 보이는 누수
> - 사전은 train으로만

> ✅ **한 줄 정리** · 단어사전은 train 토큰을 정렬해 번호를 매기고, 0 = `<pad>` · 1 = `<unk>`로 빈 칸과 처음 보는 단어를 처리한다.

**딥러닝 코드에서는**

- `vocab = {"<pad>": 0, "<unk>": 1, ...}`를 JSON으로 저장해 추론 때 그대로 사용 (Python문법 7-1)

### 1-3. `A` 패딩과 길이 정보 · `pad_sequence` · `pack_padded_sequence` <sub>보강 · RNN-1 3-4 참조</sub>

> 🎯 **길이가 다른 문장을 한 배치 텐서로 만들고, 진짜 끝을 잃지 않으려면 어떻게 하는가?**

💡 **쉽게 말하면**

- 패딩은 **길이가 다른 줄을 같은 길이 상자에 넣고 빈 칸에 0을 채우기**.
- 어디까지가 진짜인지는 상자에 없으니 길이표(`lengths`)를 함께 들고 다님.

<img src="images/rnn3_padding.png" alt="pad_sequence와 진짜 끝" width="718">

**실행 흐름**

1. 번호 텐서 3개 · `pad_sequence`
2. 같은 LSTM에 pad 그대로 · `pack_padded_sequence`로 넣기
3. 진짜 끝 비교
4. `pad_packed_sequence`로 되돌리기

**먼저 예측** · pad 그대로 넣은 LSTM의 `out[:, -1]`이 진짜 끝과 같은 줄은 몇 개일까? — 답은 코드 실행 결과 아래에

In [5]:
# [보강] 길이가 다른 3문장 → 번호 텐서 목록
texts = ["음악이 좋다", "영화가 정말 최고다",                     # 길이 2 · 3 · 6
         "결말이 별로다 하지만 음악이 아주 좋다"]
seqs = [torch.tensor(encode(t, vocab, tok_word)) for t in texts]  # 문장마다 번호 텐서
lengths = torch.tensor([len(s) for s in seqs])                    # 진짜 길이 [2, 3, 6]


# pad_sequence: 가장 긴 6칸에 맞춰 뒤를 0(<pad>)으로 채움
X = pad_sequence(seqs, batch_first=True, padding_value=0)  # 가장 긴 줄에 맞춰 0 채움
print("X", tuple(X.shape), "· lengths", lengths.tolist())  # 모양 · 길이
print("첫 줄:", X[0].tolist())                             # 뒤가 0


# 같은 LSTM에 두 방식으로 넣기
set_seed()                                                   # 모델마다 같은 시작점
emb = nn.Embedding(len(vocab), 8, padding_idx=0)             # 번호 → 8차원
lstm = nn.LSTM(8, 16, batch_first=True)                      # 같은 LSTM
e = emb(X)                                                   # (3, 6, 8)
out, _ = lstm(e)                                             # pad 칸까지 계산
packed = pack_padded_sequence(e, lengths, batch_first=True,  # pad 칸을 빼고 묶음
                              enforce_sorted=False)          # pad 칸을 건너뜀
p_out, (h_n, c_n) = lstm(packed)                             # pack으로 계산
idx = torch.arange(len(X))                                   # 줄 번호 0 · 1 · 2
real_end = out[idx, lengths - 1]                             # 줄마다 진짜 마지막 칸
print("pad 방식 out[:, -1] == 진짜 끝:",                     # 줄마다 비교
      [torch.allclose(a, b) for a, b in zip(out[:, -1], real_end)])
print("pack 방식 h_n[-1] == 진짜 끝:",                       # h_n이 진짜 끝인지
      torch.allclose(h_n[-1], real_end, atol=1e-6))


# pack 결과를 다시 (3, 6, 16)으로 펼치면 pad 칸은 0
p_out, p_len = pad_packed_sequence(p_out, batch_first=True)     # 다시 (N, T, H)로
print("되돌린 모양", tuple(p_out.shape), "· 첫 줄 pad 칸 합:",  # pad 칸은 0
      p_out[0, 2:].abs().sum().item())

X (3, 6) · lengths [2, 3, 6]
첫 줄: [13, 17, 0, 0, 0, 0]
pad 방식 out[:, -1] == 진짜 끝: [False, False, True]
pack 방식 h_n[-1] == 진짜 끝: True
되돌린 모양 (3, 6, 16) · 첫 줄 pad 칸 합: 0.0


**결과** · 예측 정답은 1개 — 가장 긴 셋째 줄만. pack하면 `h_n`이 세 줄 모두 진짜 끝

- `X (3, 6) · lengths [2, 3, 6]` — 가장 긴 6칸에 맞춰 채움, 진짜 길이는 따로
- `첫 줄: [13, 17, 0, 0, 0, 0]` — 뒤 4칸은 0(`<pad>`)
- `pad 방식 out[:, -1] == 진짜 끝: [False, False, True]` — 짧은 줄은 pad 칸을 더 지난 값을 읽음 (RNN-1 3-4와 같은 함정)
- `pack 방식 h_n[-1] == 진짜 끝: True` — pack하면 `h_n`이 줄마다 진짜 끝 → `lens - 1` 인덱싱이 필요 없음
- `되돌린 모양 (3, 6, 16) · 첫 줄 pad 칸 합: 0.0` — pad 칸은 계산하지 않아 0

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>pad_sequence</code></td>
<td style="text-align: left;">가장 긴 줄에 맞춰 0으로 채움 → 직사각형 텐서</td>
<td style="text-align: left;">배치를 만들 때 <strong>항상</strong> 먼저</td>
</tr>
<tr>
<td style="text-align: left;"><code>pack_padded_sequence</code></td>
<td style="text-align: left;">pad 칸을 건너뛰고 계산, <code>h_n</code>이 줄마다 진짜 끝</td>
<td style="text-align: left;">RNN 계열에서 마지막 시점이 필요할 때</td>
</tr>
<tr>
<td style="text-align: left;">pad만 + <code>out[idx, lens - 1]</code></td>
<td style="text-align: left;">pad 칸까지 계산하지만 진짜 끝을 직접 꺼냄</td>
<td style="text-align: left;">pack을 못 쓰는 구조 · 평균 풀링과 함께</td>
</tr>
<tr>
<td style="text-align: left;"><code>pad_packed_sequence</code></td>
<td style="text-align: left;">pack 결과를 다시 <code>(N, T, H)</code>로, pad 칸은 0</td>
<td style="text-align: left;">칸마다 출력이 필요할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `pack_padded_sequence`의 `lengths`는 CPU 텐서여야 하고, 길이순 정렬이 아니면 `enforce_sorted=False`

> ✅ **한 줄 정리** · 문장은 `pad_sequence`로 같은 길이로 채우고 `lengths`를 함께 넘겨 `pack_padded_sequence`로 pad 칸을 건너뛴다.

**딥러닝 코드에서는**

- `collate_fn`에서 `pad_sequence(batch, batch_first=True)`와 `lengths`를 함께 돌려주고, 모델 `forward(x, lengths)`에서 pack

---
## 2. 임베딩

> **이 장의 흐름**
>
> - 1장에서 번호를 얻었다
> - 번호 13과 15는 크기 차이에 뜻이 없으니, 번호마다 학습되는 짧은 벡터를 붙여 모델이 비슷함을 배우게 함 (심화학습 8장)

**파이프라인 위치** · ④ 임베딩

### 2-1. `A` one-hot vs Embedding · `nn.Embedding` <sub>심화학습 8.3 · 8.4 · 연습문제 8.1 + 보강</sub>

> 🎯 **단어 번호를 one-hot 대신 `nn.Embedding`으로 바꾸면 무엇이 줄고, 둘은 계산상 어떤 관계인가?**

💡 **쉽게 말하면**

- one-hot은 **5만 칸짜리 출석부에 한 칸만 체크**, 임베딩은 **300칸짜리 짧은 자기소개 카드**.
- 카드 내용(숫자)은 학습으로 채워짐.

<img src="images/rnn3_embedding.png" alt="Embedding = one-hot × 가중치 표" width="756">

**실행 흐름**

1. 연습문제 8.1 메모리 계산
2. 같은 번호를 `emb(ids)` · `one_hot(ids) @ weight`로 비교
3. 파라미터 수 · `<pad>` 줄

**먼저 예측** · 어휘 50,000 · 300차원이면 단어 하나에 one-hot은 임베딩의 몇 배 메모리를 쓸까? — 답은 코드 실행 결과 아래에

In [6]:
# [보강] 연습문제 8.1 계산: 어휘 50,000 · float32(4 bytes)에서 단어 하나의 메모리
V, D = 50_000, 300                                                     # 연습문제 8.1 조건
one_hot_bytes, emb_bytes = V * 4, D * 4                                # 칸 수 × 4 bytes
print(f"단어 1개 · one-hot {one_hot_bytes:,} bytes · 임베딩 {emb_bytes:,} bytes"  # 단어 하나
      f" · {one_hot_bytes / emb_bytes:.1f}배")
print(f"100단어 문장 · one-hot {one_hot_bytes * 100 / 2**20:.2f} MiB"  # 문장 하나
      f" · 임베딩 {emb_bytes * 100 / 2**10:.1f} KiB")


# nn.Embedding = 번호로 표의 한 줄을 꺼내기 = one-hot × 가중치 표
set_seed()                                                       # 모델마다 같은 시작점
emb = nn.Embedding(len(vocab), 4, padding_idx=0)                 # 사전 25 × 4차원
ids = torch.tensor(encode("음악이 정말 좋다", vocab, tok_word))  # 번호 3개
by_lookup = emb(ids)                                             # (3, 4)
one_hot = F.one_hot(ids, len(vocab)).float()                     # (3, 25)
by_onehot = one_hot @ emb.weight                                 # (3, 25) @ (25, 4) → (3, 4)
print("emb(ids) == one_hot @ weight:", torch.allclose(by_lookup, by_onehot),  # 같은 결과
      "| 모양", tuple(by_lookup.shape))
print("파라미터:", emb.weight.numel(), "= 25 × 4 · <pad> 줄:",   # 표 크기 · 0번 줄
      emb.weight[0].tolist())

단어 1개 · one-hot 200,000 bytes · 임베딩 1,200 bytes · 166.7배
100단어 문장 · one-hot 19.07 MiB · 임베딩 117.2 KiB
emb(ids) == one_hot @ weight: True | 모양 (3, 4)
파라미터: 100 = 25 × 4 · <pad> 줄: [0.0, 0.0, 0.0, 0.0]


**결과** · 예측 정답은 `166.7`배 — 200,000 bytes vs 1,200 bytes

- `단어 1개 · one-hot 200,000 bytes · 임베딩 1,200 bytes · 166.7배` — 50000 × 4 = 200000 · 300 × 4 = 1200 (연습문제 8.1 정답과 같음)
- `100단어 문장 · one-hot 19.07 MiB · 임베딩 117.2 KiB` — 원본 정답의 '19.5MB'는 계산 실수 — 2천만 bytes는 19.07 MiB (임베딩 117 KB는 맞음)
- `emb(ids) == one_hot @ weight: True | 모양 (3, 4)` — Embedding = one-hot × 가중치 표, 실제로는 곱셈 없이 번호로 줄만 꺼냄
- `파라미터: 100 = 25 × 4 · <pad> 줄: [0.0, 0.0, 0.0, 0.0]` — 사전 크기 × 차원, `padding_idx` 줄은 0으로 고정

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">one-hot</td>
<td style="text-align: left;">단어 수만큼 긴 벡터, 한 칸만 1 → 희소 · 의미 없음</td>
<td style="text-align: left;">종류가 적은 범주 (RNN-1의 숫자 0 ~ 9)</td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.Embedding</code></td>
<td style="text-align: left;">번호 → 학습되는 짧은 실수 벡터 (표에서 한 줄 꺼내기)</td>
<td style="text-align: left;">단어처럼 종류가 많을 때 (<strong>기본 선택</strong>)</td>
</tr>
<tr>
<td style="text-align: left;"><code>padding_idx=0</code></td>
<td style="text-align: left;">0번 줄은 0으로 고정, 학습되지 않음</td>
<td style="text-align: left;"><code>&lt;pad&gt;</code>가 계산에 끼어들지 않게</td>
</tr>
<tr>
<td style="text-align: left;">사전학습 임베딩 (Word2Vec · GloVe · FastText)</td>
<td style="text-align: left;">큰 말뭉치에서 미리 배운 표</td>
<td style="text-align: left;">데이터가 적을 때 초깃값으로</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `nn.Embedding`에는 정수 번호(`long`)를 넣음
> - one-hot이나 실수 텐서를 넣으면 에러

> ✅ **한 줄 정리** · `nn.Embedding`은 one-hot에 가중치 표를 곱한 것과 같은 결과를 번호 조회로 빠르게 내며, 짧은 학습 벡터라 메모리도 훨씬 적다.

**딥러닝 코드에서는**

- `self.emb = nn.Embedding(len(vocab), 128, padding_idx=0)` → `x: (N, T)` long → `self.emb(x): (N, T, 128)` → LSTM 입력

### 2-2. `B` 단어 사이 거리 · `torch.cdist` <sub>심화학습 연습문제 8.2 + 보강</sub>

> 🎯 **'king - man + woman ≈ queen' 같은 계산은 왜 임베딩에서만 되고 one-hot에서는 안 되는가?**

💡 **쉽게 말하면**

- one-hot은 **모든 학생이 서로 똑같이 떨어져 앉은 교실**이라 누가 누구와 비슷한지 표현할 수 없고, 임베딩은 비슷한 학생끼리 가까이 앉힐 수 있음.

<img src="images/rnn3_king.png" alt="손으로 정한 2차원 임베딩" width="548">

**실행 흐름**

1. one-hot 4개의 거리
2. 축에 뜻(왕족 · 성별)을 둔 2차원 벡터로 king - man + woman
3. 같은 계산을 one-hot으로

**먼저 예측** · one-hot으로 king - man + woman을 계산하면 queen과 가장 가까울까? — 답은 코드 실행 결과 아래에

In [7]:
# [보강] 연습문제 8.2 확인: one-hot 은 모든 단어 사이 거리가 같음
words = ["king", "man", "woman", "queen"]          # 연습문제 8.2 단어
oh = torch.eye(4)                                  # 단어 4개 one-hot
dist = torch.cdist(oh, oh)                         # 모든 쌍의 유클리드 거리
print("one-hot 거리 king ↔ man · woman · queen:",  # 모두 √2
      [round(d, 3) for d in dist[0, 1:].tolist()])


# 손으로 만든 2차원 임베딩: 축 0 = 왕족, 축 1 = 성별(+남 · -여)
vec = {"king": torch.tensor([0.9, 0.8]), "man": torch.tensor([0.1, 0.8]),  # 축에 뜻을 둔 벡터
       "woman": torch.tensor([0.1, -0.8]), "queen": torch.tensor([0.9, -0.8])}
q = vec["king"] - vec["man"] + vec["woman"]                                # 벡터 계산
near = min(words, key=lambda w: torch.dist(q, vec[w]).item())              # 가장 가까운 단어
print("king - man + woman =", [round(v, 2) for v in q.tolist()],           # 결과 · 가장 가까운 단어
      "→ 가장 가까운 단어:", near)


# one-hot 으로 같은 계산을 하면 어휘에 없는 벡터
q_oh = oh[0] - oh[1] + oh[2]                                       # one-hot으로 같은 계산
print("one-hot 계산 결과:", q_oh.tolist(), "→ 네 단어까지 거리:",  # queen이 가장 멂
      [round(torch.dist(q_oh, oh[i]).item(), 3) for i in range(4)])

one-hot 거리 king ↔ man · woman · queen: [1.414, 1.414, 1.414]
king - man + woman = [0.9, -0.8] → 가장 가까운 단어: queen
one-hot 계산 결과: [1.0, -1.0, 1.0, 0.0] → 네 단어까지 거리: [1.414, 2.449, 1.414, 2.0]


**결과** · 예측 정답은 아님 — queen까지 `2.0`, king · woman까지 `1.414`로 오히려 더 멂

- `one-hot 거리 king ↔ man · woman · queen: [1.414, 1.414, 1.414]` — 모든 쌍이 √2로 같음 → 비슷함을 표현 못 함
- `king - man + woman = [0.9, -0.8] → 가장 가까운 단어: queen` — 축에 뜻이 있으면 벡터 계산이 뜻 계산이 됨
- `one-hot 계산 결과: [1.0, -1.0, 1.0, 0.0] → 네 단어까지 거리: [1.414, 2.449, 1.414, 2.0]` — 결과가 어휘에 없는 벡터이고 queen이 가장 멂 (연습문제 8.2)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">one-hot 거리</td>
<td style="text-align: left;">모든 쌍이 √2로 같음</td>
<td style="text-align: left;">비슷함을 표현할 수 없음</td>
</tr>
<tr>
<td style="text-align: left;">임베딩 거리 · 코사인 유사도</td>
<td style="text-align: left;">비슷한 쓰임의 단어가 가깝게 학습됨</td>
<td style="text-align: left;">유사 단어 찾기 · 의미 계산</td>
</tr>
<tr>
<td style="text-align: left;">손으로 정한 축 (이 셀)</td>
<td style="text-align: left;">축의 뜻을 사람이 정함</td>
<td style="text-align: left;">원리 설명용 (실제로는 데이터가 정함)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 이 셀의 축 뜻은 설명용으로 사람이 정한 것
> - 실제 임베딩의 축은 데이터에서 학습되며 사람이 읽을 수 있는 뜻이 아닐 수 있음

> ✅ **한 줄 정리** · one-hot은 모든 단어가 같은 거리라 관계를 담지 못하고, 임베딩은 학습된 위치 덕분에 비슷함과 관계를 거리 · 덧셈으로 표현한다.

**딥러닝 코드에서는**

- 학습된 임베딩 비교는 `F.cosine_similarity(emb.weight[i], emb.weight[j], dim=0)`
- 비슷한 단어 찾기에 사용

---
## 3. 문장 분류 (보강)

> **이 장의 흐름**
>
> - 2장에서 단어가 벡터 줄이 됐다
> - RNN-1의 LSTM으로 읽어 문장 하나에 답 하나(긍정 · 부정)를 내고, 처음 보는 표현에서도 버티는지 확인

**파이프라인 위치** · ⑤ 순서 모델 → ⑥ 요약 · 분류 → ⑦ 평가

### 3-1. `A` 마지막 시점 vs 평균 풀링 · `h_n[-1]` · `mask 평균` <sub>보강 · RNN-1 3-4와 연결</sub>

> 🎯 **문장 분류에서 LSTM 출력을 어떻게 요약해야 하며, 순서를 버린 평균은 어디서 실패하는가?**

💡 **쉽게 말하면**

- 'A 하지만 B' 문장은 **뒤 절이 결론**.
- 단어 벡터를 섞어 평균 내면 좋은 말 하나 · 나쁜 말 하나가 섞여 어느 쪽이 결론인지 사라짐.

<img src="images/rnn3_acc.png" alt="요약 방법 · 단어 vs 글자 정확도" width="912">

**실행 흐름**

1. 분류 모델 하나에 요약 방법 3가지(순서 없는 평균 · 마지막 시점 · 평균 풀링)
2. 같은 데이터 · 설정으로 학습
3. 전체 · 단문 · 대조문 정확도

**먼저 예측** · 순서를 버린 Embedding 평균 모델은 대조문('A 하지만 B')을 몇 % 맞힐까? — 답은 코드 실행 결과 아래에

In [8]:
# [보강] 문장 분류 모델: Embedding → (LSTM) → 요약 방법 3가지 → Linear(2)
class SentClf(nn.Module):                                            # 문장 분류 모델
    def __init__(self, vocab_size, how, emb_dim=16, hid=32):         # 사전 크기 · 요약 방법
        super().__init__()                                           # nn.Module 준비 (필수)
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)  # 번호 → 벡터 (pad는 0)
        self.lstm = nn.LSTM(emb_dim, hid, batch_first=True)          # 순서대로 읽기
        self.out = nn.Linear(emb_dim if how == "순서 없는 평균" else hid, 2)  # 요약 → 긍정 · 부정
        self.how = how                                               # 방법 기억

    def forward(self, x, lengths):                      # 번호 + 길이
        e = self.emb(x)                                 # (N, T, emb_dim)
        m = (x != 0).float().unsqueeze(-1)              # 진짜 칸 = 1, pad = 0
        if self.how == "순서 없는 평균":                # 단어 벡터 평균 (LSTM 안 씀)
            return self.out((e * m).sum(1) / m.sum(1))  # 진짜 칸 평균 → 분류
        if self.how == "마지막 시점":                   # pack → 줄마다 진짜 끝의 h
            packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True,  # 길이는 CPU 텐서
                                          enforce_sorted=False)
            _, (h_n, _) = self.lstm(packed)             # 줄마다 진짜 끝의 h
            return self.out(h_n[-1])                    # 마지막 층 h → 분류
        out, _ = self.lstm(e)                           # 평균 풀링: 진짜 칸의 out 평균
        return self.out((out * m).sum(1) / m.sum(1))    # LSTM 출력 평균 → 분류

In [9]:
# 문장 목록 → 텐서 · 학습 · 평가 함수 (3-2 · 5장에서도 사용)
def make_batch(data, vocab, tok):  # 문장 목록 → (X, lengths, y, kind)
    seqs = [torch.tensor(encode(s, vocab, tok)) for s, _, _ in data]       # 문장 → 번호
    X = pad_sequence(seqs, batch_first=True)                               # 0으로 채움
    lengths = torch.tensor([len(s) for s in seqs])                         # 진짜 길이
    y = torch.tensor([lab for _, lab, _ in data])                          # 정답
    return X.to(DEV), lengths.to(DEV), y.to(DEV), [k for _, _, k in data]  # GPU로 + 종류


def train_clf(tok, how, epochs=40):                              # 사전 · 모델 · 학습을 한 번에
    voc = build_vocab([s for s, _, _ in train], tok)             # train 으로만 사전
    X, L, y, _ = make_batch(train, voc, tok)                     # 학습 텐서
    set_seed()                                                   # 모델마다 같은 시작점
    model = SentClf(len(voc), how).to(DEV)                       # 새 모델
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)          # Adam
    for _ in range(epochs):                                      # epoch 반복
        perm = torch.randperm(len(X)).to(DEV)                    # 줄 순서 섞기
        for i in range(0, len(X), 32):                           # 32문장씩
            j = perm[i:i + 32]                                   # 이번 배치
            opt.zero_grad()                                      # 지난 기울기 지우기
            F.cross_entropy(model(X[j], L[j]), y[j]).backward()  # 손실 → 기울기
            opt.step()                                           # 가중치 한 걸음 수정
    return model.eval(), voc                                     # 평가 모드 · 사전


def evaluate(model, voc, tok, data):                           # 전체 · 단문 · 대조문 정확도
    X, L, y, kinds = make_batch(data, voc, tok)                # 평가 텐서
    with torch.no_grad():                                      # 평가는 기울기 기록 안 함
        ok = (model(X, L).argmax(1) == y).float().cpu()        # 문장마다 맞았나
    pick = {k: torch.tensor([kd == k for kd in kinds]) for k in ("단문", "대조")}  # 종류별 mask
    return (ok.mean().item(), ok[pick["단문"]].mean().item(),  # 전체 · 단문 · 대조문
            ok[pick["대조"]].mean().item())

In [10]:
# 같은 데이터 · 같은 학습 설정으로 요약 방법만 바꿔 비교 (단어 단위)
models = {}                                                   # 요약 방법별 모델
print(kpad("요약 방법", 15), "전체   단문   대조문")          # 머리글
for how in ["순서 없는 평균", "마지막 시점", "평균 풀링"]:    # 세 방법
    models[how], vocab_w = train_clf(tok_word, how)           # 같은 조건 학습
    acc = evaluate(models[how], vocab_w, tok_word, test)      # 처음 보는 문장으로 평가
    print(kpad(how, 15), "  ".join(f"{a:.3f}" for a in acc))  # 정확도 셋

요약 방법       전체   단문   대조문


순서 없는 평균  0.615  1.000  0.497


마지막 시점     1.000  1.000  1.000


평균 풀링       1.000  1.000  1.000


**결과** · 예측 정답은 `0.497` — 찍기. 단문은 `1.000`

- `순서 없는 평균  0.615  1.000  0.497` — 단어 벡터 평균은 순서를 버려 대조문에서 찍기 (단문은 전부 맞힘)
- `마지막 시점     1.000  1.000  1.000` — LSTM이 '하지만' 뒤 절을 기억해 전부 맞힘
- `평균 풀링       1.000  1.000  1.000` — LSTM 출력의 평균도 전부 맞힘 — 칸마다 출력에 앞 순서가 이미 담겨 있음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">순서 없는 평균 (Embedding 평균)</td>
<td style="text-align: left;">단어 벡터만 평균, 순서를 버림</td>
<td style="text-align: left;">순서가 상관없는 주제 분류의 빠른 기준선</td>
</tr>
<tr>
<td style="text-align: left;">마지막 시점 (<code>h_n[-1]</code>)</td>
<td style="text-align: left;">끝까지 읽은 요약 하나</td>
<td style="text-align: left;">한 문장에 정답 하나, 뒤쪽이 중요할 때</td>
</tr>
<tr>
<td style="text-align: left;">평균 풀링 (mask 평균)</td>
<td style="text-align: left;">모든 칸 출력의 평균, 앞쪽 정보도 고르게</td>
<td style="text-align: left;">긴 문장 · 앞쪽 단어도 중요할 때</td>
</tr>
<tr>
<td style="text-align: left;">양방향 LSTM (<code>bidirectional=True</code>)</td>
<td style="text-align: left;">앞 · 뒤 두 방향 요약을 이어 붙임</td>
<td style="text-align: left;">문장 전체가 이미 있을 때 (RNN-1 3-3)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 평균 풀링에서 pad 칸까지 평균하면 짧은 문장일수록 0이 섞여 흐려짐
> - mask로 진짜 칸만

> ✅ **한 줄 정리** · 순서가 뜻을 바꾸는 문장은 LSTM으로 읽어야 하며, 요약은 마지막 시점(`h_n`)이나 mask 평균 풀링 중 val로 고른다.

**딥러닝 코드에서는**

- `_, (h, _) = self.lstm(packed)` → `self.fc(h[-1])`, 평균 풀링은 `(out * mask).sum(1) / mask.sum(1)`

### 3-2. `A` 처음 보는 단어 처리 · `<unk>` <sub>보강</sub>

> 🎯 **학습 때 없던 활용형('재미있었다')이 들어오면 단어 단위와 글자 단위 모델은 각각 어떻게 되는가?**

💡 **쉽게 말하면**

- 단어 단위는 **모르는 단어를 통째로 지우개로 지움**, 글자 단위는 **모르는 글자 한두 개만 지움** — '재미있?다'는 남은 글자로 뜻을 짐작할 수 있음.

**실행 흐름**

1. 글자 단위 LSTM 학습
2. unseen의 `<unk>` 비율
3. 같은 문장이 두 방식으로 어떻게 바뀌는지
4. test · unseen 정확도

**먼저 예측** · 활용형만 바뀐 unseen 문장에서 단어 단위 LSTM의 정확도는? — 답은 코드 실행 결과 아래에

In [11]:
# [보강] 처음 보는 활용형(재미있었다 · 지루했다 …) 문장에서 <unk> 비율
model_c, vocab_c = train_clf(tok_char, "마지막 시점")  # 글자 단위 LSTM
for name, voc, tok in [("단어", vocab_w, tok_word), ("글자", vocab_c, tok_char)]:  # 두 사전
    X, _, _, _ = make_batch(unseen, voc, tok)          # 활용형 문장 → 번호
    unk = ((X == 1).sum() / (X != 0).sum()).item()     # 진짜 칸 중 <unk> 비율
    print(f"{name} 단위 · 사전 {len(voc)}개 · unseen 의 <unk> 비율 {unk:.3f}")  # 비교


# 같은 문장이 어떻게 바뀌나
s = "음악이 정말 재미있었다"                                               # 활용형 문장
print("단어 →", [w if w in vocab_w else "<unk>" for w in tok_word(s)])     # 단어가 통째로 사라짐
print("글자 →", "".join(c if c in vocab_c else "?" for c in tok_char(s)))  # 모르는 글자만 ?


# 정확도: 학습에 없던 문장(test) · 처음 보는 활용형(unseen)
print(kpad("마지막 시점 LSTM", 18), "test 전체  unseen 전체 · 단문 · 대조문")       # 머리글
for name, m, voc, tok in [("단어 단위", models["마지막 시점"], vocab_w, tok_word),  # 두 모델
                          ("글자 단위", model_c, vocab_c, tok_char)]:
    t = evaluate(m, voc, tok, test)[0]                                              # test 전체
    u = evaluate(m, voc, tok, unseen)                                               # unseen 셋
    print(kpad(name, 18), f"{t:.3f}     ", " · ".join(f"{a:.3f}" for a in u))       # 결과

단어 단위 · 사전 25개 · unseen 의 <unk> 비율 0.283
글자 단위 · 사전 49개 · unseen 의 <unk> 비율 0.110
단어 → ['음악이', '정말', '<unk>']
글자 → 음악이정말재미있?다
마지막 시점 LSTM   test 전체  unseen 전체 · 단문 · 대조문
단어 단위          1.000      0.533 · 0.550 · 0.517


글자 단위          1.000      1.000 · 1.000 · 1.000


**결과** · 예측 정답은 `0.533` — 찍기 수준. 글자 단위는 `1.000`

- `단어 단위 · 사전 25개 · unseen 의 <unk> 비율 0.283` — 평가 토큰의 28.3%가 사전에 없음 (평가 단어 전부)
- `글자 단위 · 사전 49개 · unseen 의 <unk> 비율 0.110` — 처음 보는 글자(았 · 었 · 했 · 웠 등)만 `<unk>`
- `단어 → ['음악이', '정말', '<unk>']` — 평가 단어가 통째로 사라짐
- `글자 → 음악이정말재미있?다` — '재미있'은 남아 뜻을 짐작할 수 있음
- `단어 단위          1.000      0.533 · 0.550 · 0.517` — 익숙한 문장은 전부 맞히지만 활용형은 찍기
- `글자 단위          1.000      1.000 · 1.000 · 1.000` — 활용형도 전부 맞힘

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">단어 단위</td>
<td style="text-align: left;">처음 보는 단어 → 통째로 <code>&lt;unk&gt;</code></td>
<td style="text-align: left;">어휘가 고정된 데이터</td>
</tr>
<tr>
<td style="text-align: left;">글자 단위</td>
<td style="text-align: left;">처음 보는 단어도 아는 글자로 대부분 남음</td>
<td style="text-align: left;">활용 · 오타가 많은 한국어</td>
</tr>
<tr>
<td style="text-align: left;">사전 크게 만들기</td>
<td style="text-align: left;"><code>&lt;unk&gt;</code>는 줄지만 드문 단어는 학습 데이터가 부족</td>
<td style="text-align: left;">말뭉치가 아주 클 때</td>
</tr>
<tr>
<td style="text-align: left;">최소 빈도 컷 (<code>min_freq</code>)</td>
<td style="text-align: left;">드문 단어를 일부러 <code>&lt;unk&gt;</code>로 → <code>&lt;unk&gt;</code>도 학습됨</td>
<td style="text-align: left;">실제 텍스트 프로젝트 (보통 2 ~ 5)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - test 정확도만 보면 두 모델이 똑같이 `1.000`
> - 처음 보는 표현이 섞인 평가 데이터를 따로 만들어 봐야 차이가 드러남

> ✅ **한 줄 정리** · 처음 보는 단어는 `<unk>`가 되어 정보를 잃으므로, 활용이 많은 한국어는 글자 · 부분 단어 단위가 더 강하다.

**딥러닝 코드에서는**

- 실제 텍스트 프로젝트는 부분 단어 토크나이저 + 드문 단어 `<unk>` 처리(`min_freq`), 평가에는 신조어
- 오타 문장을 따로 넣어 확인

---
## 4. Attention 맛보기 (보강)

> **이 장의 흐름**
>
> - 3장의 LSTM은 단어를 차례로 전달했다
> - 강의가 이름만 짚은 Attention을 한 번 실행해 '한 번에 모두 보기'가 무엇인지 모양으로 확인

**파이프라인 위치** · ⑤ 순서 모델 (다른 방식)

### 4-1. `B` Attention 맛보기 · `nn.MultiheadAttention` <sub>보강 · 강의 '오늘 안 다룬 것' (Seq2Seq · Attention · Transformer)</sub>

> 🎯 **Attention은 RNN과 무엇이 다르며, `nn.MultiheadAttention`은 무엇을 돌려주는가?**

💡 **쉽게 말하면**

- RNN은 **줄 서서 귓속말 전달**(멀리 갈수록 흐려짐), Attention은 **회의실에서 모두가 서로를 한 번에 봄** — 누구를 얼마나 볼지 비율(가중치)을 계산.

<img src="images/rnn3_attention.png" alt="RNN과 Attention" width="712">

**실행 흐름**

1. 두 문장 패딩
2. 3-1 모델의 학습된 단어 벡터
3. `MultiheadAttention(16, 2)`에 Q = K = V로 넣기 (pad mask)
4. 모양 · 가중치 합 · pad 가중치
5. 한 단어의 가중치

**한 줄 식** · `가중치 = softmax(Q · Kᵀ ÷ √d)` → `출력 = 가중치 · V` (Q · K · V는 같은 단어 벡터에 각각 다른 Linear를 곱한 것)

**먼저 예측** · 가중치 텐서 `(2, 6, 6)`에서 한 줄(한 단어가 다른 단어들을 보는 비율)의 합은? — 답은 코드 실행 결과 아래에

In [12]:
# [보강] Attention 맛보기: 문장 안 모든 단어가 서로를 한 번에 봄
texts = ["음악이 좋다", "결말이 별로다 하지만 음악이 아주 좋다"]    # 길이 2 · 6
seqs = [torch.tensor(encode(t, vocab_w, tok_word)) for t in texts]  # 번호 텐서
X = pad_sequence(seqs, batch_first=True)                            # (2, 6)
pad_mask = X == 0                                                   # True = <pad> 칸 (보지 말 것)


# 3-1 마지막 시점 모델의 학습된 단어 벡터 + 처음 만든 Attention 층
set_seed()                                                 # 모델마다 같은 시작점
e = models["마지막 시점"].emb(X.to(DEV)).detach()          # (2, 6, 16)
attn = nn.MultiheadAttention(embed_dim=16, num_heads=2,    # Attention 층 (head 2개)
                             batch_first=True).to(DEV)
out, w = attn(e, e, e, key_padding_mask=pad_mask.to(DEV))  # Q = K = V
print("입력", tuple(e.shape), "→ out", tuple(out.shape),   # 모양
      "· 가중치", tuple(w.shape))
print("한 줄 가중치 합:", round(w[0, 0].sum().item(), 4),  # 합 1 · pad 0
      "| 첫 문장의 pad 칸 가중치 합:", w[0, :, 2:].sum().item())


# 둘째 문장 마지막 단어 '좋다'가 각 단어를 보는 비율 (두 head 평균)
row = w[1, 5].cpu()  # 마지막 단어가 보는 비율
print(" · ".join(f"{itos[i]} {v:.2f}" for i, v in zip(X[1].tolist(), row)))  # 단어별 비율

입력 (2, 6, 16) → out (2, 6, 16) · 가중치 (2, 6, 6)
한 줄 가중치 합: 1.0 | 첫 문장의 pad 칸 가중치 합: 0.0
결말이 0.06 · 별로다 0.33 · 하지만 0.13 · 음악이 0.21 · 아주 0.16 · 좋다 0.13


**결과** · 예측 정답은 `1.0` — softmax 비율. pad 칸은 mask로 `0.0`

- `입력 (2, 6, 16) → out (2, 6, 16) · 가중치 (2, 6, 6)` — 칸마다 6칸 모두를 보는 비율 표 (6 × 6), 출력 모양은 입력과 같음
- `한 줄 가중치 합: 1.0 | 첫 문장의 pad 칸 가중치 합: 0.0` — softmax라 합 1, `key_padding_mask`로 pad 칸은 0
- `결말이 0.06 · 별로다 0.33 · 하지만 0.13 · 음악이 0.21 · 아주 0.16 · 좋다 0.13` — 마지막 '좋다'가 문장 전체를 한 번에 봄 (Attention 층은 학습 전이라 비율 크기에 뜻은 없음)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">RNN 계열 (LSTM · GRU)</td>
<td style="text-align: left;">한 칸씩 차례로, 먼 단어는 여러 칸을 거쳐 전달</td>
<td style="text-align: left;">짧은 문장 · 시계열 · 계산 자원이 적을 때</td>
</tr>
<tr>
<td style="text-align: left;">Attention</td>
<td style="text-align: left;">모든 칸이 모든 칸을 한 번에 봄, 칸끼리 병렬 계산</td>
<td style="text-align: left;">긴 문장 · 먼 단어 관계가 중요할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>key_padding_mask</code></td>
<td style="text-align: left;"><code>True</code>인 칸(pad)은 가중치 0</td>
<td style="text-align: left;">패딩된 배치에서 <strong>항상</strong></td>
</tr>
<tr>
<td style="text-align: left;">Transformer</td>
<td style="text-align: left;">Attention 층을 쌓은 구조 (BERT · GPT)</td>
<td style="text-align: left;">현재 텍스트 모델의 기본 (다음 과정)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `key_padding_mask`를 빼면 pad 칸에도 가중치가 가서 결과가 길이에 따라 흔들림

> ✅ **한 줄 정리** · Attention은 모든 칸이 모든 칸을 한 번에 보는 비율(가중치)로 섞는 층이며, pad 칸은 `key_padding_mask`로 막는다.

**딥러닝 코드에서는**

- Transformer 블록 = `MultiheadAttention` + `Linear` 층을 쌓은 것, BERT
- GPT 같은 사전학습 모델이 이 구조 (다음 과정)

---
## 5. 종합 실습 · 새 문장을 예측하는 텍스트 분류 파이프라인 (보강)

> **이 장의 흐름**
>
> - 1 ~ 3장을 한 흐름으로 묶음
> - 3-2에서 처음 보는 활용형에 강했던 **글자 단위**로 처음부터 끝까지 실행

**파이프라인 위치** · ① 토큰화 → ② 사전 → ③ 번호 · 패딩 → ④ 임베딩 → ⑤ LSTM → ⑥ 분류 → ⑦ 평가 · 새 문장 예측

> 🎯 **새 문장을 넣으면 긍정 · 부정을 답하는 파이프라인을 처음부터 끝까지 만들려면 무엇이 필요한가?**

💡 **쉽게 말하면**

- 번역기에 문장을 넣는 것처럼, **학습 때와 똑같은 순서**(같은 토큰화 · 같은 사전 · 같은 패딩)로 새 문장을 바꿔 넣어야 모델이 알아봄.

<img src="images/rnn3_pipeline.png" alt="텍스트 분류 파이프라인 7단계" width="748">

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1 · 2 · 3</td>
<td style="text-align: left;">글자 토큰 → train 사전 → 번호 · 패딩 텐서</td>
<td style="text-align: left;">1-1 <code>tok_char</code> · 1-2 <code>build_vocab</code> · 3-1 <code>make_batch</code></td>
</tr>
<tr>
<td style="text-align: left;">4 · 5</td>
<td style="text-align: left;">Embedding → LSTM(pack, 마지막 시점) → Linear 학습</td>
<td style="text-align: left;">3-1 <code>train_clf</code></td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">test · unseen 평가</td>
<td style="text-align: left;">3-1 <code>evaluate</code></td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">새 문장 → 같은 사전으로 번호 → 예측 확률</td>
<td style="text-align: left;"><code>predict()</code> · <code>F.softmax</code></td>
</tr>
</tbody>
</table>

**먼저 예측** · 학습 때 한 번도 본 적 없는 '화면이 아주 좋았다 하지만 줄거리가 너무 지루했다'는 긍정일까 부정일까? — 답은 코드 실행 결과 아래에

In [13]:
# [보강] 1단계 토큰화 · 2단계 train 사전 · 3단계 번호 · 패딩 (글자 단위)
voc = build_vocab([s for s, _, _ in train], tok_char)                    # 글자 사전
X, L, y, _ = make_batch(train, voc, tok_char)                            # 번호 · 패딩
print(f"1 · 2 · 3단계 → 사전 {len(voc)}개 · 학습 텐서 {tuple(X.shape)}"  # 확인
      f" · 길이 {L.min().item()}~{L.max().item()}")


# 4 · 5단계: Embedding → LSTM(pack, 마지막 시점) → Linear 학습
model, voc = train_clf(tok_char, "마지막 시점")                                    # 학습
n_par = sum(p.numel() for p in model.parameters())                                 # 파라미터 수
print(f"4 · 5단계 → 파라미터 {n_par:,}개 (Embedding {model.emb.weight.numel()})")  # 확인


# 6단계: 평가 (학습에 없던 문장 · 처음 보는 활용형)
print("6단계 → test", f"{evaluate(model, voc, tok_char, test)[0]:.3f}",  # 두 평가
      "· unseen", f"{evaluate(model, voc, tok_char, unseen)[0]:.3f}")


# 7단계: 새 문장 예측 함수
def predict(text):                                               # 새 문장 → 긍정 · 부정
    x = torch.tensor([encode(text, voc, tok_char)], device=DEV)  # 같은 사전으로 번호
    with torch.no_grad():                                        # 평가는 기울기 기록 안 함
        prob = F.softmax(model(x, torch.tensor([x.shape[1]])), dim=1)[0, 1]  # 긍정 확률
    return ("긍정" if prob > 0.5 else "부정"), prob.item()       # 결과 · 확률


for s in ["배우 연기가 정말 훌륭했다",                   # 처음 보는 문장 4개
          "화면이 아주 좋았다 하지만 줄거리가 너무 지루했다",
          "줄거리가 조금 지루했다 하지만 음악이 정말 감동적이었다",
          "결말이 꽤 실망스러웠다"]:
    label, p = predict(s)                                # 예측
    print(f"7단계 → {label} (긍정 확률 {p:.3f}) | {s}")  # 결과

1 · 2 · 3단계 → 사전 49개 · 학습 텐서 (600, 26) · 길이 6~26


4 · 5단계 → 파라미터 7,250개 (Embedding 784)


6단계 → test 1.000 · unseen 1.000
7단계 → 긍정 (긍정 확률 1.000) | 배우 연기가 정말 훌륭했다
7단계 → 부정 (긍정 확률 0.000) | 화면이 아주 좋았다 하지만 줄거리가 너무 지루했다
7단계 → 긍정 (긍정 확률 1.000) | 줄거리가 조금 지루했다 하지만 음악이 정말 감동적이었다
7단계 → 부정 (긍정 확률 0.000) | 결말이 꽤 실망스러웠다


**결과** · 예측 정답은 부정 — 뒤 절('지루했다')이 결론, 활용형 '좋았다' · '지루했다'는 처음 보지만 글자 단위라 읽어 냄

- `1 · 2 · 3단계 → 사전 49개 · 학습 텐서 (600, 26) · 길이 6~26` — 특수 토큰 2개 + 글자 47개, 가장 긴 26글자에 맞춰 패딩
- `4 · 5단계 → 파라미터 7,250개 (Embedding 784)` — Embedding 49 × 16 = 784, LSTM 4 × 1600 = 6400, Linear 32 × 2 + 2 = 66
- `6단계 → test 1.000 · unseen 1.000` — 처음 보는 문장 · 처음 보는 활용형 모두 맞힘
- `7단계 → 부정 (긍정 확률 0.000)` — 화면이 아주 좋았다 하지만 줄거리가 너무 지루했다 → 앞 절은 긍정, 뒤 절이 결론
- `7단계 → 긍정 (긍정 확률 1.000)` — 줄거리가 조금 지루했다 하지만 음악이 정말 감동적이었다 → 반대 순서

**해석**

- 이 데이터는 규칙이 단순해 확률이 0 · 1로 딱 떨어짐
- 실제 후기 문장은 표현이 훨씬 다양해 확률이 중간값으로 나옴
- 같은 파이프라인에 데이터 · 토크나이저만 바꿔 씀

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">학습 때와 같은 사전 · 토큰화로 예측</td>
<td style="text-align: left;">같은 번호 체계</td>
<td style="text-align: left;"><strong>항상</strong> (사전 · 설정을 모델과 함께 저장)</td>
</tr>
<tr>
<td style="text-align: left;">새 문장으로 사전을 다시 만들기</td>
<td style="text-align: left;">번호가 바뀌어 모델이 엉뚱한 벡터를 읽음</td>
<td style="text-align: left;">절대 하지 않음</td>
</tr>
<tr>
<td style="text-align: left;">확률(<code>softmax</code>)로 보기</td>
<td style="text-align: left;">얼마나 확신하는지까지</td>
<td style="text-align: left;">애매한 문장 걸러내기 · 임곗값 조정</td>
</tr>
<tr>
<td style="text-align: left;">argmax만 보기</td>
<td style="text-align: left;">긍정 · 부정 결과만</td>
<td style="text-align: left;">정확도 계산</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `predict()`는 문장 하나라 패딩이 없음
> - 여러 문장을 한 번에 넣을 때는 `make_batch`처럼 `lengths`를 함께 넘겨야 함

> ✅ **한 줄 정리** · 토큰화 → train 사전 → 번호 · 패딩 → Embedding → LSTM → 분류를 같은 순서로 학습 · 예측에 쓰고, 처음 보는 표현이 섞인 평가로 확인한다.

**딥러닝 코드에서는**

- `torch.save({"model": model.state_dict(), "vocab": vocab}, "clf.pt")`
- 모델 가중치와 사전을 **한 파일에 함께** 저장해 추론 때 같은 번호 체계를 씀

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. 토큰화 · 사전</td>
<td style="text-align: left;">단어 23종 · 글자 47종, train 사전 25개, 처음 보는 '좋았다' → 1(<code>&lt;unk&gt;</code>)</td>
<td style="text-align: left;">1-1 · 1-2</td>
</tr>
<tr>
<td style="text-align: left;">2. 패딩 · pack</td>
<td style="text-align: left;"><code>X (3, 6)</code>, pad 방식 진짜 끝 <code>[False, False, True]</code> vs pack <code>True</code></td>
<td style="text-align: left;">1-3</td>
</tr>
<tr>
<td style="text-align: left;">3. Embedding</td>
<td style="text-align: left;"><code>emb(ids) == one_hot @ weight: True</code>, 연습문제 8.1 → 166.7배</td>
<td style="text-align: left;">2장</td>
</tr>
<tr>
<td style="text-align: left;">4. 순서 · 처음 보는 단어</td>
<td style="text-align: left;">순서 없는 평균 대조문 <code>0.497</code>, unseen 단어 단위 <code>0.533</code> vs 글자 단위 <code>1.000</code></td>
<td style="text-align: left;">3장</td>
</tr>
<tr>
<td style="text-align: left;">5. Attention</td>
<td style="text-align: left;">가중치 <code>(2, 6, 6)</code>, 한 줄 합 <code>1.0</code>, pad 칸 <code>0.0</code></td>
<td style="text-align: left;">4-1</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">토큰화</td>
<td style="text-align: left;">단어 · 글자 · 부분 단어, 순서가 뜻을 바꿈</td>
<td style="text-align: left;"><code>tokenizer(text)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">단어사전</td>
<td style="text-align: left;">train으로만, 0 = <code>&lt;pad&gt;</code> · 1 = <code>&lt;unk&gt;</code></td>
<td style="text-align: left;"><code>vocab.get(w, 1)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">패딩</td>
<td style="text-align: left;"><code>pad_sequence</code> + <code>lengths</code> → pack</td>
<td style="text-align: left;"><code>collate_fn</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">임베딩</td>
<td style="text-align: left;">번호 → 학습 벡터 = one-hot × W</td>
<td style="text-align: left;"><code>nn.Embedding(V, D, padding_idx=0)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">문장 분류</td>
<td style="text-align: left;">순서가 중요하면 LSTM, 요약은 <code>h_n[-1]</code> · mask 평균</td>
<td style="text-align: left;"><code>self.fc(h_n[-1])</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">처음 보는 단어</td>
<td style="text-align: left;"><code>&lt;unk&gt;</code>는 정보 손실, 글자 · 부분 단어가 강함</td>
<td style="text-align: left;">신조어 평가 문장 따로</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">예측</td>
<td style="text-align: left;">학습 때와 같은 토큰화 · 사전</td>
<td style="text-align: left;">모델 + 사전 함께 저장</td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">단어 거리</td>
<td style="text-align: left;">one-hot 거리 모두 √2, 임베딩은 관계 표현</td>
<td style="text-align: left;"><code>F.cosine_similarity</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">Attention</td>
<td style="text-align: left;">모두가 모두를 한 번에, pad는 mask</td>
<td style="text-align: left;"><code>nn.MultiheadAttention</code></td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">단어 단위와 글자 단위 토큰화는 무엇이 다른가?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">단어사전의 0번 · 1번은 무엇이며, 사전은 어느 데이터로 만드는가?</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;"><code>pad_sequence</code>와 <code>pack_padded_sequence</code>는 각각 무엇을 하는가?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;"><code>nn.Embedding</code>과 one-hot은 계산상 어떤 관계이며, 무엇이 줄어드는가?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">one-hot으로는 왜 king - man + woman ≈ queen이 안 되는가?</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">'A 하지만 B' 문장에서 순서 없는 평균은 왜 실패하며, LSTM 출력은 어떻게 요약하는가?</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">처음 보는 활용형에 단어 단위와 글자 단위는 어떻게 다르게 반응하는가?</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">Attention은 RNN과 무엇이 다르며, pad 칸은 어떻게 막는가?</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">새 문장을 예측할 때 반드시 학습 때와 같아야 하는 것은?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

학습 없이 텍스트 → 텐서 변환만 확인하는 문제. **먼저 손으로 답을 적고** 아래 빈 셀에서 코드로 확인.

1. `"나는 밥을 먹었다"`의 단어 토큰 수와 글자 토큰 수(공백 제외)
2. train 문장 `["a b", "b c d"]`로 `<pad>` 0 · `<unk>` 1 · 나머지 정렬 순서 사전을 만들고 `"a e"`를 번호로
3. 번호 줄 `[2, 3]` · `[3, 4, 5]`를 `pad_sequence(batch_first=True)`로 묶은 결과와 `lengths`
4. `nn.Embedding(10, 4, padding_idx=0)`의 파라미터 수와 0번 벡터
5. 어휘 20,000 · 128차원 · float32에서 단어 하나의 one-hot · 임베딩 bytes와 배율

In [14]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. train 문장 `["나는 밥 먹었다", "나는 빵 먹었다"]`로 사전(`<pad>` 0 · `<unk>` 1 · 정렬 순서)을 만들면 크기는? `"나는 국 먹었다"`의 번호는?
2. 어휘 30,000 · 128차원 `nn.Embedding(30000, 128, padding_idx=0)`의 파라미터 수는?
3. `nn.MultiheadAttention(32, 4, batch_first=True)`에 `(5, 12, 32)`를 Q = K = V로 넣으면 출력 · 가중치 모양은?

In [15]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">단어 단위는 짧고 뜻이 분명하지만 종류가 많고 처음 보는 단어에 약함, 글자 단위는 길지만 종류가 적고 처음 보는 단어도 글자로 남음</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">0 = <code>&lt;pad&gt;</code>(빈 칸), 1 = <code>&lt;unk&gt;</code>(사전에 없는 단어). train 데이터로만</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;"><code>pad_sequence</code>는 0으로 채워 직사각형 텐서, <code>pack_padded_sequence</code>는 <code>lengths</code>로 pad 칸을 건너뛰어 <code>h_n</code>이 진짜 끝</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;"><code>emb(ids) == one_hot(ids) @ weight</code>, 곱셈 없이 줄만 꺼냄. 어휘 수 차원 → 수백 차원으로 줄어듦</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">one-hot은 모든 단어 사이 거리가 √2로 같아 관계를 담지 못하고, 계산 결과도 어휘에 없는 벡터</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">단어 평균은 순서를 버려 어느 절이 결론인지 모름(찍기). <code>h_n[-1]</code> 또는 mask 평균 풀링</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">단어 단위는 통째로 <code>&lt;unk&gt;</code>가 돼 찍기 수준, 글자 단위는 모르는 글자만 빠져 대부분 맞힘</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">모든 칸이 모든 칸을 한 번에 봄(가중치 합 1). <code>key_padding_mask</code>로 pad 칸 가중치 0</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">토큰화 방법 · 단어사전(번호) · 패딩 방식 — 모델과 사전을 함께 저장</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
# 1. 단어 · 글자 토큰 수
s = "나는 밥을 먹었다"
print(len(s.split()), len(s.replace(" ", "")))  # 3 7

# 2. train 으로만 사전 → 처음 보는 e 는 1
words = sorted({w for t in ["a b", "b c d"] for w in t.split()})
vocab = {"<pad>": 0, "<unk>": 1}
vocab.update({w: i + 2 for i, w in enumerate(words)})
print(vocab)  # {'<pad>': 0, '<unk>': 1, 'a': 2, 'b': 3, 'c': 4, 'd': 5}
print([vocab.get(w, 1) for w in "a e".split()])  # [2, 1]

# 3. 0 으로 채우고 길이는 따로
seqs = [torch.tensor([2, 3]), torch.tensor([3, 4, 5])]
X = pad_sequence(seqs, batch_first=True)
print(X.tolist(), [len(q) for q in seqs])  # [[2, 3, 0], [3, 4, 5]] [2, 3]

# 4. 파라미터 = 10 × 4, padding_idx 줄은 0
emb = nn.Embedding(10, 4, padding_idx=0)
zero = emb(torch.tensor([0])).tolist()  # <pad> 벡터
print(emb.weight.numel(), zero)  # 40 [[0.0, 0.0, 0.0, 0.0]]

# 5. bytes = 칸 수 × 4
print(20000 * 4, 128 * 4, 20000 / 128)  # 80000 512 156.25
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">크기 6 (<code>나는</code> 2 · <code>먹었다</code> 3 · <code>밥</code> 4 · <code>빵</code> 5) · <code>[2, 1, 3]</code> (<code>국</code>은 처음 보는 단어 → 1)</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">30000 × 128 = 3840000개 (<code>&lt;pad&gt;</code> 줄도 자리는 차지, 학습만 안 됨)</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">출력 <code>(5, 12, 32)</code> (입력과 같음) · 가중치 <code>(5, 12, 12)</code> (head 평균)</td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) 패딩한 배치에서 <code>out[:, -1]</code>로 분류</td>
<td style="text-align: left;">모든 줄 길이가 같다고 착각</td>
<td style="text-align: left;"><code>lengths</code> + pack의 <code>h_n[-1]</code></td>
<td style="text-align: left;">이 배치에 <code>&lt;pad&gt;</code>가 있나?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. 텍스트를 RNN에 넣기까지 어떤 전처리를 하나요?**
> - 문장을 토큰으로 나누고, 학습 데이터로만 단어사전을 만들어 번호로 바꿉니다.
> - 사전에는 패딩용 `<pad>`와 처음 보는 단어용 `<unk>`를 넣어 둡니다.
> - 길이가 다른 문장은 `pad_sequence`로 채우고 실제 길이를 함께 넘겨 `pack_padded_sequence`로 패딩을 건너뛰게 한 뒤, `nn.Embedding`으로 벡터로 바꿔 LSTM에 넣습니다.

**Q2. one-hot 대신 임베딩을 쓰는 이유는?**
> - one-hot은 어휘 수만큼 긴 희소 벡터라 메모리가 크고, 모든 단어 사이 거리가 같아 비슷함을 표현하지 못합니다.
> - 임베딩은 수백 차원의 학습 벡터라 어휘 5만
> - 300차원 기준으로 단어 하나의 메모리가 약 167분의 1이고, 비슷한 단어가 가까워지도록 학습됩니다.
> - 계산상으로는 one-hot에 가중치 행렬을 곱한 것과 같고, 실제로는 번호로 행을 조회합니다.

**Q3. 처음 보는 단어(OOV)는 어떻게 처리하나요?**
> - 단어사전에 없으면 `<unk>`로 바꾸는데, 그러면 그 단어의 정보가 사라집니다.
> - 실험에서 활용형만 바꾼 문장에 단어 단위 모델은 찍기 수준이었고 글자 단위 모델은 대부분 맞혔습니다.
> - 그래서 활용이 많은 한국어는 글자나 부분 단어 단위 토크나이저를 쓰고, 드문 단어를 일부러 `<unk>`로 바꿔 학습시키기도 합니다.

**Q4. RNN 계열과 Attention의 차이는?**
> - RNN 계열은 단어를 차례로 읽어 정보를 한 칸씩 넘기므로 먼 단어의 정보가 여러 단계를 거치고 병렬 계산이 어렵습니다.
> - Attention은 모든 단어가 모든 단어를 한 번에 보고 가중치로 섞어 먼 관계도 한 단계에 닿고 병렬로 계산됩니다.
> - 패딩 칸은 `key_padding_mask`로 가중치가 0이 되게 막고, 이 층을 쌓은 것이 Transformer입니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>str.split()</code> · <code>list()</code> · <code>str.replace()</code></td>
<td style="text-align: left;">str · Python 내장 함수</td>
<td style="text-align: left;">단어 · 글자 단위로 나누기</td>
<td style="text-align: left;">1-1 <code>text.split()</code> · <code>list(text.replace(...))</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>sorted({...})</code> · <code>enumerate()</code> · <code>dict.update()</code></td>
<td style="text-align: left;">Python 내장 함수 · dict</td>
<td style="text-align: left;">정렬된 사전 · 번호 매기기</td>
<td style="text-align: left;">1-2 <code>sorted({w for s in texts for w in tok(s)})</code> · <code>vocab.update(...)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>dict.get(키, 기본값)</code></td>
<td style="text-align: left;">dict</td>
<td style="text-align: left;">없는 단어 → <code>&lt;unk&gt;</code></td>
<td style="text-align: left;">1-2 <code>vocab.get(w, vocab[...])</code> (없으면 <code>&lt;unk&gt;</code>)</td>
</tr>
<tr>
<td style="text-align: left;"><code>pad_sequence(batch_first=True, padding_value=0)</code></td>
<td style="text-align: left;">torch.nn.utils.rnn</td>
<td style="text-align: left;">0으로 채워 직사각형 텐서</td>
<td style="text-align: left;">1-3 <code>X = pad_sequence(seqs, batch_first=True, padding_v…</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>pack_padded_sequence(lengths, enforce_sorted=False)</code></td>
<td style="text-align: left;">torch.nn.utils.rnn</td>
<td style="text-align: left;">pad 칸 건너뛰기</td>
<td style="text-align: left;">1-3 <code>packed = pack_padded_sequence(e, lengths, batch_fi…</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>pad_packed_sequence()</code></td>
<td style="text-align: left;">torch.nn.utils.rnn</td>
<td style="text-align: left;">pack 결과를 다시 펼치기</td>
<td style="text-align: left;">1-3 <code>p_out, p_len = pad_packed_sequence(p_out, batch_fi…</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.Embedding(V, D, padding_idx=0)</code></td>
<td style="text-align: left;">torch.nn</td>
<td style="text-align: left;">번호 → 학습 벡터</td>
<td style="text-align: left;">2-1 <code>emb = nn.Embedding(len(vocab), 4, padding_idx=0)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>F.one_hot(ids, V)</code> · <code>@</code></td>
<td style="text-align: left;">torch.nn.functional · 연산자</td>
<td style="text-align: left;">one-hot · 행렬 곱 (비교용)</td>
<td style="text-align: left;">2-1 <code>one_hot = F.one_hot(ids, len(vocab)).float()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>torch.eye()</code> · <code>torch.cdist()</code> · <code>torch.dist()</code></td>
<td style="text-align: left;">torch</td>
<td style="text-align: left;">단위 행렬 · 모든 쌍 거리 · 두 벡터 거리</td>
<td style="text-align: left;">2-2 <code>oh = torch.eye(4)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>min(words, key=...)</code></td>
<td style="text-align: left;">Python 내장 함수</td>
<td style="text-align: left;">가장 가까운 단어 고르기</td>
<td style="text-align: left;">2-2 <code>near = min(words, key=lambda w: torch.dist(q, vec[…</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.Embedding</code> → <code>nn.LSTM</code> → <code>nn.Linear</code></td>
<td style="text-align: left;">torch.nn</td>
<td style="text-align: left;">문장 분류 모델</td>
<td style="text-align: left;">3-1 <code>self.emb = nn.Embedding(vocab_size, emb_dim, paddi…</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>(x != 0).float().unsqueeze(-1)</code></td>
<td style="text-align: left;">Tensor 비교 · 메서드</td>
<td style="text-align: left;">진짜 칸 mask</td>
<td style="text-align: left;">3-1 <code>m = (x != 0).float().unsqueeze(-1)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>h_n[-1]</code> · <code>(out * m).sum(1) / m.sum(1)</code></td>
<td style="text-align: left;">인덱싱 · Tensor 연산</td>
<td style="text-align: left;">마지막 시점 · 평균 풀링</td>
<td style="text-align: left;">3-1 <code>_, (h_n, _) = self.lstm(packed)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>F.cross_entropy</code> · <code>torch.optim.Adam</code></td>
<td style="text-align: left;">torch</td>
<td style="text-align: left;">분류 손실 · 최적화</td>
<td style="text-align: left;">3-1 <code>F.cross_entropy(model(X[j], L[j]), y[j]).backward()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)</code></td>
<td style="text-align: left;">torch.nn</td>
<td style="text-align: left;">Attention 층</td>
<td style="text-align: left;">4-1 <code>attn = nn.MultiheadAttention(embed_dim=16, num_hea…</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>attn(q, k, v, key_padding_mask=...)</code></td>
<td style="text-align: left;">nn.MultiheadAttention 호출</td>
<td style="text-align: left;">출력 · 가중치</td>
<td style="text-align: left;">4-1 <code>attn(e, e, e, key_padding_mask=pad_mask.to(DEV))</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.detach()</code></td>
<td style="text-align: left;">Tensor 메서드</td>
<td style="text-align: left;">기울기 기록 끊기 (관찰용)</td>
<td style="text-align: left;">4-1 <code>e = models["마지막 시점"].emb(X.to(DEV)).detach()</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">사전은 train 문장으로만</td>
<td style="text-align: left;">test까지 합쳐 사전 만들기</td>
</tr>
<tr>
<td style="text-align: left;"><code>padding_idx=0</code> · mask로 pad 제외</td>
<td style="text-align: left;">pad 칸까지 평균 · <code>out[:, -1]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>lengths</code>를 함께 넘겨 pack</td>
<td style="text-align: left;">길이 정보 버리기</td>
</tr>
<tr>
<td style="text-align: left;">학습에 있던 문장은 평가에서 빼기</td>
<td style="text-align: left;">같은 문장으로 채점</td>
</tr>
<tr>
<td style="text-align: left;">처음 보는 표현이 섞인 평가 데이터 따로</td>
<td style="text-align: left;">test 하나만 보고 끝내기</td>
</tr>
<tr>
<td style="text-align: left;">모델과 사전을 함께 저장</td>
<td style="text-align: left;">예측 때 사전을 새로 만들기</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">단어 단위 vs 글자 단위 토큰</td>
<td style="text-align: left;">뜻 단위 · 종류 많음 vs 글자 단위 · 처음 보는 단어에 강함</td>
<td style="text-align: left;">활용이 많은 한국어는 글자 · 부분 단어</td>
</tr>
<tr>
<td style="text-align: left;"><code>&lt;pad&gt;</code> vs <code>&lt;unk&gt;</code></td>
<td style="text-align: left;">길이 맞춤용 빈 칸 vs 사전에 없는 단어</td>
<td style="text-align: left;">pad는 계산에서 빼고, unk는 학습함</td>
</tr>
<tr>
<td style="text-align: left;">train 사전 vs 전체 사전</td>
<td style="text-align: left;">test 단어 차단 vs test 단어가 미리 들어감</td>
<td style="text-align: left;">train으로만</td>
</tr>
<tr>
<td style="text-align: left;"><code>pad_sequence</code> vs <code>pack_padded_sequence</code></td>
<td style="text-align: left;">채워서 모양 맞추기 vs pad 칸 건너뛰기</td>
<td style="text-align: left;">채운 뒤 pack, 둘은 함께 씀</td>
</tr>
<tr>
<td style="text-align: left;"><code>out[:, -1]</code> vs <code>h_n[-1]</code>(pack)</td>
<td style="text-align: left;">틀의 끝 vs 줄마다 진짜 끝</td>
<td style="text-align: left;">패딩 배치면 pack의 <code>h_n</code></td>
</tr>
<tr>
<td style="text-align: left;">one-hot vs <code>nn.Embedding</code></td>
<td style="text-align: left;">길고 희소 · 거리 모두 같음 vs 짧고 학습됨</td>
<td style="text-align: left;">단어는 Embedding</td>
</tr>
<tr>
<td style="text-align: left;"><code>emb(ids)</code> vs <code>one_hot(ids) @ W</code></td>
<td style="text-align: left;">줄 꺼내기 vs 행렬 곱</td>
<td style="text-align: left;">결과는 같고 Embedding이 훨씬 빠름</td>
</tr>
<tr>
<td style="text-align: left;">순서 없는 평균 vs LSTM</td>
<td style="text-align: left;">순서 버림 vs 순서대로 읽음</td>
<td style="text-align: left;">'하지만'처럼 순서가 뜻을 바꾸면 LSTM</td>
</tr>
<tr>
<td style="text-align: left;">마지막 시점 vs 평균 풀링</td>
<td style="text-align: left;">끝 요약 하나 vs 모든 칸 평균</td>
<td style="text-align: left;">둘 다 시도해 val로 고름</td>
</tr>
<tr>
<td style="text-align: left;">RNN 계열 vs Attention</td>
<td style="text-align: left;">차례로 전달 vs 한 번에 모두 보기</td>
<td style="text-align: left;">긴 문장 · 큰 데이터는 Attention(Transformer)</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개를 새로 풀기 · 1-2 단어사전 함수를 빈 셀에 다시 짜기</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">내가 쓴 문장 20개로 5장 파이프라인을 처음부터 다시 실행</td>
<td style="text-align: left;">40분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 다시 풀기 · RNN-1 · 2 · 3의 ✅ 줄만 이어 읽기</td>
<td style="text-align: left;">20분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 심화학습 노트북 8장(자연어 처리와 임베딩 · 연습문제 8.1 · 8.2)과 강의의 텍스트 관련 부분을 뼈대로 함
- 수업 자료에 없는 텍스트 코드(토큰화 → 사전 → 번호 → 패딩 → Embedding → LSTM 분류 → Attention)를 **보강**
- 4개 대단원 · 8개 섹션 + 5장 종합 실습(새 문장 예측), 데이터는 노트북 안에서 만든 영화 후기 문장
- RNN · LSTM 구조와 패딩 기본은 RNN-1, 기준선 · 분할은 RNN-2 참조로 대신함

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
<tr>
<td style="text-align: left;">보강</td>
<td style="text-align: left;">수업 자료에 없는 내용 — 수업에 텍스트 코드가 없어 <strong>코드는 모두 보강</strong> (코드 첫 줄 <code># [보강]</code>, 섹션 출처 <code>보강</code>)</td>
</tr>
</tbody>
</table>

**섹션 구성** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드 → 예측 정답 · 결과 해설 → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리

**코드 주석** · 코드 위 = 이 단계에서 하는 일, 코드 옆 = 실행 흐름 · 문법 구조

**실행 환경** · 해설의 수치는 **GPU(CUDA) · torch 2.11 · seed 고정**으로 두 번 실행해 같은 값이 나온 것. GPU가 없으면 CPU로 실행되며 학습 결과 숫자가 조금 다를 수 있음

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">수업 자료</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">섹션</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">심화학습 8.1 · 8.2</td>
<td style="text-align: left;">자연어 처리 · 단어를 숫자로 바꿔야 하는 이유</td>
<td style="text-align: left;">1-1 · 1-2</td>
</tr>
<tr>
<td style="text-align: left;">심화학습 8.3 · 8.4 · 연습문제 8.1</td>
<td style="text-align: left;">one-hot의 문제 · 임베딩 · 메모리 계산</td>
<td style="text-align: left;">2-1 (원본 정답의 '19.5MB' 계산 실수 바로잡음)</td>
</tr>
<tr>
<td style="text-align: left;">심화학습 연습문제 8.2</td>
<td style="text-align: left;">king - man + woman ≈ queen · one-hot 거리</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">강의 '순서가 중요하다'</td>
<td style="text-align: left;">개가 사람을 물었다 / 사람이 개를 물었다 · 재료는 같고 뜻은 다름</td>
<td style="text-align: left;">1-1 · 3-1</td>
</tr>
<tr>
<td style="text-align: left;">강의 padding · Bidirectional</td>
<td style="text-align: left;">길이가 다른 줄 · 양방향</td>
<td style="text-align: left;">1-3 · 3-1 비교표 (기본은 RNN-1 3-3 · 3-4 참조)</td>
</tr>
<tr>
<td style="text-align: left;">강의 '오늘 안 다룬 것'</td>
<td style="text-align: left;">Seq2Seq · Attention · Transformer 이름</td>
<td style="text-align: left;">4-1</td>
</tr>
<tr>
<td style="text-align: left;">(보강)</td>
<td style="text-align: left;">토큰화 · 단어사전 · pad · unk · pack · LSTM 분류 · Attention 실행 · 새 문장 예측</td>
<td style="text-align: left;">1-1 ~ 4-1 · 5장</td>
</tr>
</tbody>
</table>

심화학습 1 ~ 7장 · 9장 종합문제(RNN · LSTM · GRU · BPTT)는 RNN-1과 같은 내용이라 **다시 다루지 않음**.

### 전체 지도

#### 단원 연결 · 왜 이 순서로 배우는가

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">순서</th>
<th style="text-align: left;">배우는 이유</th>
<th style="text-align: left;">대단원</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;">1</td>
<td style="text-align: left;">모델은 글자를 못 읽으니 문장을 번호 텐서로 바꾼다</td>
<td style="text-align: left;">문장을 숫자로</td>
</tr>
<tr>
<td style="text-align: center;">2</td>
<td style="text-align: left;">번호 자체에는 뜻이 없으니 학습되는 벡터로 바꾼다</td>
<td style="text-align: left;">임베딩</td>
</tr>
<tr>
<td style="text-align: center;">3</td>
<td style="text-align: left;">벡터 줄을 LSTM으로 읽어 문장 하나의 답을 낸다</td>
<td style="text-align: left;">문장 분류</td>
</tr>
<tr>
<td style="text-align: center;">4</td>
<td style="text-align: left;">차례로 읽지 않고 한 번에 보는 방법을 맛본다</td>
<td style="text-align: left;">Attention 맛보기</td>
</tr>
<tr>
<td style="text-align: center;">5</td>
<td style="text-align: left;">1 ~ 3을 묶어 새 문장을 예측한다</td>
<td style="text-align: left;">종합 실습</td>
</tr>
</tbody>
</table>

#### 텍스트 분류 파이프라인 · 각 대단원이 쓰이는 곳

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰이는 장</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">① 토큰화</td>
<td style="text-align: left;">문장 → 단어 · 글자 조각</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">② 단어사전</td>
<td style="text-align: left;">train 토큰 → 번호, <code>&lt;pad&gt;</code> · <code>&lt;unk&gt;</code></td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">③ 번호 · 패딩</td>
<td style="text-align: left;">문장 → 번호 목록 → <code>pad_sequence</code> + <code>lengths</code></td>
<td style="text-align: left;">1-3</td>
</tr>
<tr>
<td style="text-align: left;">④ 임베딩</td>
<td style="text-align: left;">번호 → 학습 벡터 <code>(N, T, D)</code></td>
<td style="text-align: left;">2장</td>
</tr>
<tr>
<td style="text-align: left;">⑤ 순서 모델</td>
<td style="text-align: left;">LSTM(pack) 또는 Attention</td>
<td style="text-align: left;">3-1 · 4-1</td>
</tr>
<tr>
<td style="text-align: left;">⑥ 요약 · 분류</td>
<td style="text-align: left;">마지막 시점 · 평균 풀링 → Linear</td>
<td style="text-align: left;">3-1</td>
</tr>
<tr>
<td style="text-align: left;">⑦ 평가 · 예측</td>
<td style="text-align: left;">처음 보는 문장 · 표현으로 확인, 새 문장 예측</td>
<td style="text-align: left;">3-2 · 5장</td>
</tr>
</tbody>
</table>

### 전체 코드

텍스트 분류의 핵심(문장 데이터 → 단어사전 → 패딩 → Embedding → LSTM(pack) → test 평가)을 장별 셀로 모음.

- **위에서부터 이 셀들만 실행해도 동작**
- 각 출력 앞 `[번호]`가 장

In [16]:
# [보강] 문장 → 토큰 → 사전 → 패딩 → Embedding → LSTM 분류 (한 셀 요약)
import random  # 파이썬 난수 (데이터 만들기)

import torch                                                       # 텐서 · 자동 미분
import torch.nn as nn                                              # 층: RNN · LSTM · GRU · Linear
import torch.nn.functional as F                                    # 함수: one_hot · cross_entropy
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence  # 패딩 · pack

SEED = 20260912                            # 수업 코드와 같은 seed
torch.backends.cudnn.deterministic = True  # GPU에서도 매번 같은 계산 결과
torch.backends.cudnn.benchmark = False     # 실행마다 계산 방법이 바뀌지 않게
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU 없으면 CPU

In [17]:
# ---------- 0. 문장 데이터 ----------
SUBJ = ["영화가", "배우 연기가", "음악이", "줄거리가", "결말이", "화면이"]  # 주어 6개
ADV = ["정말", "너무", "꽤", "아주", "조금"]                                # 꾸밈말 5개
POS = ["좋다", "재미있다", "훌륭하다", "최고다", "감동적이다"]              # 긍정 평가
NEG = ["별로다", "지루하다", "최악이다", "실망스럽다", "아쉽다"]            # 부정 평가


def make_sentences(n, seed):   # 문장 데이터
    rng = random.Random(seed)  # 이 데이터 전용 난수 (전역 seed와 분리)

    def clause(good):                                          # 절 하나 만들기
        word = rng.choice(POS if good else NEG)                # 평가 단어
        return f"{rng.choice(SUBJ)} {rng.choice(ADV)} {word}"  # 주어 + 꾸밈말 + 평가

    data = []                                                        # (문장, 정답, 종류) 목록
    for _ in range(n):                                               # 줄 n개 만들기
        y = rng.randint(0, 1)                                        # 정답을 먼저 정함
        if rng.random() < 0.5:                                       # 반은 대조문
            data.append((f"{clause(1 - y)} 하지만 {clause(y)}", y))  # 대조문
        else:
            data.append((clause(y), y))                              # 단문
    return data                                                      # 문장 목록


train = make_sentences(600, seed=1)                                  # 학습 600문장
seen = {t for t, _ in train}                                         # 학습 문장
test = [d for d in make_sentences(300, seed=2) if d[0] not in seen]  # 처음 보는 문장만

In [18]:
# ---------- 1. 토큰화 → 단어사전(<pad> 0 · <unk> 1) → 번호 → 패딩 ----------
words = sorted({w for s, _ in train for w in s.split()})                     # train 단어 정렬
vocab = {"<pad>": 0, "<unk>": 1, **{w: i + 2 for i, w in enumerate(words)}}  # 사전


def batch(data):                                                # 문장 → 텐서
    seqs = [torch.tensor([vocab.get(w, 1) for w in s.split()])  # 번호 (없으면 1)
            for s, _ in data]
    X = pad_sequence(seqs, batch_first=True)                    # 0으로 채움
    L = torch.tensor([len(s) for s in seqs])                    # 길이 (CPU)
    y = torch.tensor([lab for _, lab in data])                  # 정답
    return X.to(DEV), L, y.to(DEV)                              # 입력 · 길이 · 정답


print("[1] 사전", len(vocab), "· train", len(train), "· test", len(test))  # 확인

[1] 사전 25 · train 600 · test 200


In [19]:
# ---------- 2 · 3. Embedding → LSTM(pack, 마지막 시점) → 분류 ----------
class SentClf(nn.Module):                                            # 문장 분류 모델
    def __init__(self, vocab_size, emb_dim=16, hid=32):              # 사전 크기
        super().__init__()                                           # nn.Module 준비 (필수)
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)  # 번호 → 벡터 (pad는 0)
        self.lstm = nn.LSTM(emb_dim, hid, batch_first=True)          # 순서대로 읽기
        self.out = nn.Linear(hid, 2)                                 # 긍정 · 부정

    def forward(self, x, lengths):                                             # 번호 + 길이
        packed = pack_padded_sequence(self.emb(x), lengths, batch_first=True,  # pad 칸 건너뛰기
                                      enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)                                        # 줄마다 진짜 끝의 h
        return self.out(h_n[-1])                                               # 마지막 층 h → 분류


random.seed(SEED)                                                # 파이썬 random
torch.manual_seed(SEED)                                          # torch 난수
model = SentClf(len(vocab)).to(DEV)                              # 모델
opt = torch.optim.Adam(model.parameters(), lr=1e-2)              # Adam
X, L, y = batch(train)                                           # 학습 텐서
for epoch in range(40):                                          # 40 epoch
    perm = torch.randperm(len(X))                                # 문장 순서 섞기
    for i in range(0, len(X), 32):                               # 32문장씩
        j = perm[i:i + 32]                                       # 이번 배치
        opt.zero_grad()                                          # 지난 기울기 지우기
        F.cross_entropy(model(X[j.to(DEV)], L[j]), y[j.to(DEV)]).backward()  # 손실 → 기울기
        opt.step()                                               # 가중치 한 걸음 수정
model.eval()                                                     # 평가 모드
Xt, Lt, yt = batch(test)                                         # 평가 텐서
with torch.no_grad():                                            # 평가는 기울기 기록 안 함
    acc = (model(Xt, Lt).argmax(1) == yt).float().mean().item()  # 정확도
print(f"[3] 단어 단위 LSTM · test 정확도 {acc:.3f}")             # 결과

[3] 단어 단위 LSTM · test 정확도 1.000
